# Лабораторная работа №2
## Коллекции и управление программой: платёжный шлюз

**Аудиторное время:** 4 академических часа  
**Самостоятельная работа:** 2–3 часа  
**Пререквизиты:** лабораторная №1  
**Максимум:** 20 баллов

### Итог работы

Вы обработаете пакет платёжных событий: найдёте дубликаты и
некорректные записи, нормализуете данные, примените правила
платёжного шлюза, измените балансы, сформируете очередь ручной
проверки и построите отчёт смены. В финале нужно самостоятельно
реализовать второй сценарий с дополнительным velocity-лимитом.

### Новые инструменты

`list`, `tuple`, `dict`, `set`, индексы и срезы, `for`, `while`,
`if`/`elif`/`else`, `match`, `break`, `continue`, comprehensions,
`enumerate`, методы коллекций и сортировка.

Функции, классы, файлы, NumPy и Pandas пока не используются. Это
намеренное ограничение: цель работы — увидеть состояние алгоритма
и движение данных внутри циклов.

### Правило выполнения

Запускайте ячейки сверху вниз и не изменяйте исходные наборы
`raw_events` и `capstone_events`. Перед сдачей выполните **Restart
Kernel and Run All Cells**. Все `assert` должны пройти.


<!-- clarity-self-checks-v2 -->
### Самопроверки и подсказки

Самопроверки в этом ноутбуке контролируют **контракт** решения: типы,
структуру, порядок, допустимые статусы и инварианты. Они намеренно не показывают
готовые балансы, списки решений и числовые ответы. Полная проверка выполняется
преподавателем отдельно.

В сложных заданиях есть раскрываемые подсказки двух уровней. Сначала попробуйте
решить задачу самостоятельно, затем откройте только первую подсказку. Вторая
подсказка содержит каркас алгоритма, но не готовый код и не итоговые значения.
Изменять или удалять `assert` нельзя.


## 1. Выбор коллекции

| Структура | Что хранит | Когда выбирать |
|---|---|---|
| `list` | упорядоченные элементы, повторы разрешены | очередь и журнал |
| `tuple` | неизменяемую последовательность | границы и составной ключ |
| `dict` | пары ключ → значение | баланс по номеру счёта |
| `set` | уникальные элементы | дубликаты и стоп-листы |

В этой работе одна и та же операция представлена словарём, пакет
операций — списком, стоп-лист — множеством, а границы пакета —
кортежем.


<!-- theory-formulas-v1 -->
### Теория: свойства коллекций и стоимость операций

Коллекцию выбирают по требуемым свойствам. Список моделирует последовательность

$$
L=(x_0,x_1,\ldots,x_{n-1}),
$$

множество — набор уникальных значений $S=\{x_1,\ldots,x_k\}$, словарь —
отображение $M:key\mapsto value$. Кортеж похож на список, но после создания не
изменяется.

Практическое следствие выбора:

| Операция | `list` | `set` / `dict` в среднем |
|---|---:|---:|
| взять по индексу | $O(1)$ | индекса нет |
| проверить наличие | $O(n)$ | $O(1)$ |
| добавить в конец | $O(1)$ амортизированно | $O(1)$ |
| сохранить повторы и порядок | да | множество — нет |

Запись `b = a` создаёт второе имя того же изменяемого объекта:

$$
a\longrightarrow object\longleftarrow b.
$$

`a.copy()` создаёт новый внешний контейнер. Это неглубокая копия: вложенные
изменяемые объекты всё ещё могут быть общими. В текущих событиях вложенных
контейнеров нет, поэтому копии словаря достаточно.


In [48]:
initial_balances = {
    "A100": 25_000.0,
    "A200": 15_000.0,
    "A300": 7_000.0,
    "A400": 5_000.0,
}
daily_limits = {
    "A100": 12_000.0,
    "A200": 8_000.0,
    "A300": 5_000.0,
    "A400": 3_000.0,
}
blocked_accounts = {"A400"}
blocked_merchants = {"SCAM-SHOP"}

raw_events = [
    {"id": "E001", "kind": "deposit", "account": "A100", "amount": 5000.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A100", "amount": 3200.0,
     "merchant": " books ", "city": "moscow", "counterparty": None},
    {"id": "E003", "kind": "purchase", "account": "A200", "amount": 6500.0,
     "merchant": "tech", "city": " kazan ", "counterparty": None},
    {"id": "E004", "kind": "purchase", "account": "A200", "amount": 2500.0,
     "merchant": "CAFE", "city": "Kazan", "counterparty": None},
    {"id": "E005", "kind": "withdrawal", "account": "A300", "amount": 1500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E006", "kind": "purchase", "account": "A400", "amount": 1000.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E007", "kind": "refund", "account": "A100", "amount": 700.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E008", "kind": "purchase", "account": "A100", "amount": 450.0,
     "merchant": "SCAM-SHOP", "city": "Moscow", "counterparty": None},
    {"id": "E009", "kind": "transfer", "account": "A300", "amount": 4000.0,
     "merchant": None, "city": "Moscow", "counterparty": "A200"},
    {"id": "E010", "kind": "cashout", "account": "A300", "amount": 500.0,
     "merchant": None, "city": "Moscow", "counterparty": None},
    {"id": "E002", "kind": "purchase", "account": "A300", "amount": 50.0,
     "merchant": "CAFE", "city": "Moscow", "counterparty": None},
    {"id": "E012", "kind": "purchase", "account": "A100", "amount": -200.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
    {"id": "E013", "kind": "purchase", "account": "A999", "amount": 100.0,
     "merchant": "BOOKS", "city": "Moscow", "counterparty": None},
]

# В записях только скалярные значения, поэтому для контроля
# неизменности достаточно скопировать каждый внутренний словарь.
raw_snapshot = [event.copy() for event in raw_events]
print(f"Получено событий: {len(raw_events)}")


Получено событий: 13


### Задание 1. Инвентаризация пакета — 1 балл

Не изменяя `raw_events`, получите:

- `event_ids` — список ID в исходном порядке;
- `unique_event_ids` — множество уникальных ID;
- `duplicate_ids` — множество повторяющихся ID;
- `accounts_seen` — множество всех встреченных счетов;
- `batch_bounds` — кортеж из первого и последнего ID;
- `first_three_ids` — первые три ID с помощью среза.

Здесь намеренно используйте обычные циклы: важно увидеть, как
постепенно заполняется изменяемая коллекция.


**Уточнение контракта.** Работайте именно с `raw_events` и учитывайте все
записи, в том числе те, которые позднее будут признаны некорректными. Порядок
элементов в списках должен совпадать с порядком входного пакета. Для данного
набора `raw_events` гарантированно не пуст.


In [49]:
event_ids = []
unique_event_ids = set()
duplicate_ids = set()
accounts_seen = set()
batch_bounds = None
first_three_ids = []
# YOUR CODE HERE


In [50]:
for dicts in raw_events:
    event_ids.append(dicts['id'])
    if dicts['id'] in unique_event_ids:
        duplicate_ids.add(dicts['id'])
    else:
        unique_event_ids.add(dicts['id'])
    accounts_seen.add(dicts['account'])
    batch_bounds = tuple((event_ids[0::len(raw_events)-1]))
    first_three_ids = event_ids[0:3]
print(event_ids)
print(len(event_ids))

assert type(event_ids) is list
assert len(event_ids) == len(raw_events)
assert event_ids == [event["id"] for event in raw_events]
assert unique_event_ids == set(event_ids)

expected_duplicates = {
    event_id
    for event_id in unique_event_ids
    if event_ids.count(event_id) > 1
}
assert duplicate_ids == expected_duplicates
assert accounts_seen == {event["account"] for event in raw_events}
assert type(batch_bounds) is tuple and len(batch_bounds) == 2
assert batch_bounds == (event_ids[0], event_ids[-1])
assert first_three_ids == event_ids[:3]
assert raw_events == raw_snapshot
print("Задание 1: структура и инварианты соблюдены")


['E001', 'E002', 'E003', 'E004', 'E005', 'E006', 'E007', 'E008', 'E009', 'E010', 'E002', 'E012', 'E013']
13
Задание 1: структура и инварианты соблюдены


## 2. Контроль качества данных

Внешний пакет нельзя сразу применять к балансам. Сначала нужно
отделить структурные ошибки от бизнес-решений. Оператор `continue`
завершает текущую итерацию и переходит к следующей записи — это
удобно для отбраковки.

Не удаляйте элементы из списка во время `for`: индексы сдвигаются,
и часть записей легко пропустить. Вместо этого формируйте новый
список `clean_events`.


<!-- theory-formulas-v1 -->
### Теория: валидационный конвейер и инварианты

Пусть $R$ — исходная последовательность событий, $V(e)$ — результат проверок,
а $N(e)$ — нормализация. Тогда корректный поток можно описать как

$$
C=[N(e)\mid e\in R,\;V(e)=True].
$$

Некорректные записи не исчезают бесследно: они переходят в диагностический
поток

$$
E=[(id(e),reason(e))\mid V(e)=False].
$$

Если для каждой исходной записи выбран ровно один исход, выполняется инвариант

$$
|R|=|C|+|E|.
$$

Порядок проверок определяет `reason`. При стратегии «первая причина» событие с
дубликатом ID и отрицательной суммой получит причину `duplicate_id`, если эта
проверка стоит первой. Это часть контракта, а не случайная деталь реализации.

`continue` выражает схему guard clause: обнаружили брак, записали диагностику и
перешли к следующей записи. Основной путь остаётся без глубокой вложенности.


### Задание 2. Валидация и нормализация — 3 балла

Обойдите `raw_events` по порядку. Для каждой записи примените первую
подходящую причину брака:

1. повторный ID → `duplicate_id`;
2. `amount` не является обычным `int`/`float` или не положителен →
   `invalid_amount`;
3. счёт отсутствует в `initial_balances` → `unknown_account`;
4. у перевода неизвестный получатель → `unknown_counterparty`.

Некорректные записи добавляйте в `invalid_events` в формате
`{"id": ..., "reason": ...}`. Корректную запись сначала копируйте,
затем приведите `kind` к нижнему регистру, а `merchant` и `city` — к
верхнему регистру; во всех случаях удалите внешние пробелы. `None` у merchant должен остаться `None`.

Заполните `seen_ids`, `clean_events` и `invalid_events`. Исходный
пакет должен остаться неизменным.


**Уточнение контракта.** Значение считается обычным числом только при
`type(amount) in (int, float)`; `bool` необходимо отклонить, хотя формально он
является подклассом `int`. Каждый впервые встреченный ID сразу резервируется в
`seen_ids`, даже если сама запись затем отклоняется. Поэтому повтор того же ID
позже всегда получает `duplicate_id`.

Порядок `clean_events` и `invalid_events` должен соответствовать порядку
исходного пакета. Копия корректного события должна быть новым словарём.

<details>
<summary>Подсказка 1 — структура одной итерации</summary>

В начале итерации задайте `reason = None`. Сначала выясните, встречался ли ID,
затем зарезервируйте первое появление. После этого последовательно проверяйте
сумму, счёт и получателя перевода. При найденной причине добавьте диагностику и
используйте `continue`.

</details>

<details>
<summary>Подсказка 2 — нормализация без изменения входа</summary>

После успешных проверок создайте `cleaned = event.copy()`. Меняйте только
`cleaned`: `kind` нормализуется в нижний регистр, `merchant` и `city` — в
верхний. Перед строковыми методами отдельно обработайте `merchant is None`.

</details>


In [51]:
seen_ids = set()
clean_events = []
invalid_events = []
# YOUR CODE HERE


In [52]:
allowed_input_reasons = {
    "duplicate_id",
    "invalid_amount",
    "unknown_account",
    "unknown_counterparty",
}



#Отсортировка на корректные и некорректные записи
for dicts in raw_events:
    reason = None
    if dicts['id'] in seen_ids: #проверяем id
        reason = 'duplicate_id'
    else:
        seen_ids.add(dicts['id'])
    if reason is None:
        if type(dicts['amount']) not in (int, float) or dicts['amount'] <= 0 or type(dicts['amount']) is bool: #если сумма не соответствует критериям
            reason = 'invalid_amount'
    if reason is None:
        if dicts['account'] not in initial_balances: #если счет отсутствует в initial_balances
            reason = 'unknown_account'
    if reason is None and dicts['kind'] == 'transfer': #проверка на тип операции и отсутствие получателя перевода
        if dicts['counterparty'] not in initial_balances:
            reason = 'unknown_counterparty'
    if reason is not None:
        invalid_events.append({'id':dicts['id'], 'reason':reason})
        continue

    cleaned = dicts.copy()
    cleaned['id'] = dicts['id']
    cleaned['kind'] = dicts['kind'].strip().lower()
    cleaned['city'] = dicts['city'].strip().upper()

    if cleaned['merchant'] is not None:
        cleaned['merchant'] = dicts['merchant'].strip().upper()

    clean_events.append(cleaned)
print("Задание 2: граница данных и неизменность входа соблюдены")
print(len(clean_events), len(invalid_events), len(raw_events))

assert len(clean_events) + len(invalid_events) == len(raw_events)
assert seen_ids == {event.get("id") for event in raw_events}
assert len({event["id"] for event in clean_events}) == len(clean_events)
assert all(type(event["amount"]) in (int, float) for event in clean_events)
assert all(event["amount"] > 0 for event in clean_events)
assert all(event["account"] in initial_balances for event in clean_events)
assert all(event["kind"] == event["kind"].strip().lower() for event in clean_events)
assert all(event["city"] == event["city"].strip().upper() for event in clean_events)
assert all(
    event["merchant"] is None
    or event["merchant"] == event["merchant"].strip().upper()
    for event in clean_events
)
assert all(set(item) == {"id", "reason"} for item in invalid_events)
assert all(item["reason"] in allowed_input_reasons for item in invalid_events)
assert all(
    all(cleaned is not original for original in raw_events)
    for cleaned in clean_events
)
assert raw_events == raw_snapshot

Задание 2: граница данных и неизменность входа соблюдены
10 3 13


## 3. Comprehensions и производные коллекции

Comprehension подходит для короткого преобразования или отбора без
сложного состояния. Если внутри требуется несколько веток,
изменение баланса или объяснение причины решения, обычный цикл
читается лучше.

Общая форма: `[выражение for элемент in источник if условие]`.
Аналогично создаются множества и словари.


<!-- theory-formulas-v1 -->
### Теория: отображение и фильтрация

Comprehension объединяет две идеи. Отображение применяет преобразование $f$ к
каждому элементу, фильтрация оставляет элементы, удовлетворяющие предикату $P$:

$$
Y=[f(x)\mid x\in X,\;P(x)].
$$

Соответствия синтаксису Python:

```python
[f(x) for x in source if predicate(x)]       # список
{f(x) for x in source if predicate(x)}       # множество
{key(x): value(x) for x in source}           # словарь
```

Список сохраняет порядок и повторы, множество удаляет повторы, словарь требует
уникальных ключей. Comprehension хорош, когда преобразование помещается в одну
понятную мысль. Если нужно менять несколько накопителей, выбирать причины или
управлять `break`/`continue`, обычный цикл обычно яснее.

В задании частотный словарь заполняется циклом, потому что новое значение
зависит от предыдущего:

$$
count_{t+1}(k)=count_t(k)+1.
$$


### Задание 3. Операционный обзор — 2 балла

По `clean_events` создайте:

- `active_accounts` — отсортированный список незаблокированных
  счетов с положительным балансом;
- `merchant_catalog` — множество непустых merchant;
- `high_value_ids` — ID событий на сумму не меньше 5000 в исходном
  порядке;
- `kind_counts` — словарь количества событий каждого вида.

Первые три результата получите comprehensions. Для `kind_counts`
сначала создайте словарь с нулями, затем накопите значения циклом.


**Уточнение контракта.** `active_accounts` строится по
`initial_balances` с учётом `blocked_accounts`. Остальные три результата
строятся только по `clean_events`. Ключи `kind_counts` должны совпадать с реально
встреченными нормализованными видами операций; сумма всех счётчиков равна числу
корректных событий.


In [53]:
active_accounts = []
merchant_catalog = set()
high_value_ids = []
kind_counts = {}
# YOUR CODE HERE


In [54]:
initial_balances = {
    "A100": 25_000.0,
    "A200": 15_000.0,
    "A300": 7_000.0,
    "A400": 5_000.0,
}

#Решение
for dicts in clean_events:
    active_accounts = list(dict.fromkeys(dicts['account'] for dicts in clean_events if dicts['account'] not in blocked_accounts))
    active_accounts = sorted(active_accounts)
    merchant_catalog = {dicts['merchant'] for dicts in clean_events if dicts['merchant'] is not None}
    high_value_ids = [dicts['id'] for dicts in clean_events if dicts['amount'] >= 5000]
    counter = 0
    kind_counts = {dicts['kind']: 0 for dicts in clean_events}
for dicts in clean_events:
    kind_counts[dicts['kind']] += 1
print(active_accounts)
print(kind_counts)

assert active_accounts == sorted(active_accounts)
assert active_accounts == [
    account
    for account, balance in initial_balances.items()
    if balance > 0 and account not in blocked_accounts
]
assert merchant_catalog == {
    event["merchant"]
    for event in clean_events
    if event["merchant"] is not None
}
assert high_value_ids == [
    event["id"]
    for event in clean_events
    if event["amount"] >= 5_000
]
assert set(kind_counts) == {event["kind"] for event in clean_events}
assert sum(kind_counts.values()) == len(clean_events)
assert all(
    count == sum(event["kind"] == kind for event in clean_events)
    for kind, count in kind_counts.items()
)
print("Задание 3: производные коллекции согласованы с clean_events")


['A100', 'A200', 'A300']
{'deposit': 1, 'purchase': 5, 'withdrawal': 1, 'refund': 1, 'transfer': 1, 'cashout': 1}
Задание 3: производные коллекции согласованы с clean_events


## 4. Состояние и порядок правил

Теперь каждая операция зависит от результатов предыдущих. Это
**состояние-зависимый алгоритм**: перестановка событий может изменить
итоговые балансы и решения.

Порядок проверок является частью спецификации:

1. заблокированный счёт;
2. запрещённый merchant;
3. поддерживаемый вид операции;
4. дневной лимит для списаний;
5. достаточный баланс;
6. применение операции.

`match` определяет смысл вида операции, а `if` проверяет правила.
Сумма перевода списывается у отправителя и зачисляется получателю.
Операции со статусом `REVIEW` и `REJECTED` пока не меняют баланс.


<!-- theory-formulas-v1 -->
### Теория: переходы состояния и инварианты

Перед обработкой события $t$ состояние можно представить парой

$$
X_t=(B_t,S_t),
$$

где $B_t(a)$ — баланс счёта $a$, $S_t(a)$ — накопленный дневной расход. Для
одобренного списания суммы $m$:

$$
B_{t+1}(a)=B_t(a)-m,\qquad
S_{t+1}(a)=S_t(a)+m.
$$

Для пополнения или возврата:

$$
B_{t+1}(a)=B_t(a)+m,\qquad S_{t+1}(a)=S_t(a).
$$

Для REVIEW или REJECTED состояние не меняется:

$$
X_{t+1}=X_t.
$$

Перевод $a\to b$ одновременно даёт

$$
B_{t+1}(a)=B_t(a)-m,\qquad B_{t+1}(b)=B_t(b)+m,
$$

поэтому общая сумма внутренних балансов сохраняется. Это полезный инвариант:
$\sum_a B_{t+1}(a)=\sum_a B_t(a)$ для чистого перевода.

Лимит проверяется по проектируемому расходу $S_t(a)+m$, до изменения словаря.
Правила применяются сверху вниз, а первый сработавший запрет определяет причину.


### Задание 4. Процессор платёжных событий — 4 балла

Создайте независимые копии балансов и нулевой расход по каждому
счёту. Обработайте все `clean_events`.

Для каждого события добавьте в `decisions` словарь с ключами:
`id`, `account`, `kind`, `amount`, `status`, `reason`,
`balance_after`.

Возможные статусы: `APPROVED`, `REVIEW`, `REJECTED`. Причины:
`ok`, `blocked_account`, `blocked_merchant`, `unsupported_kind`,
`daily_limit`, `insufficient_funds`.

События `REVIEW` копируйте в `review_queue`. Дневной расход
увеличивают только одобренные `purchase`, `withdrawal`, `transfer`.
`deposit` и `refund` увеличивают баланс и не расходуют лимит.


Соответствие результата и причины:

| Условие | `status` | `reason` |
|---|---|---|
| операция применена | `APPROVED` | `ok` |
| превышен дневной лимит | `REVIEW` | `daily_limit` |
| счёт заблокирован | `REJECTED` | `blocked_account` |
| merchant запрещён | `REJECTED` | `blocked_merchant` |
| вид операции не поддерживается | `REJECTED` | `unsupported_kind` |
| недостаточно средств | `REJECTED` | `insufficient_funds` |

Лимит нарушен только при строгом превышении: значение, в точности равное
`daily_limits[account]`, разрешено. `balance_after` — баланс отправителя после
обработки события; для `REVIEW` и `REJECTED` он равен текущему неизменённому
балансу. Получатель перевода получает деньги только при `APPROVED`.

<details>
<summary>Подсказка 1 — разделите направление и решение</summary>

Через `match` определите `credit`, `debit` или `unsupported`. После этого
отдельной цепочкой условий выберите статус. Не меняйте баланс до завершения всех
проверок.

</details>

<details>
<summary>Подсказка 2 — состояние одной операции</summary>

Для каждого события сначала создайте локальные `status`, `reason` и
`direction`. Ветви могут менять балансы и расход, но словарь решения удобнее
добавлять один раз в конце итерации. В очередь помещайте `event.copy()`.

</details>


In [55]:
final_balances = initial_balances.copy()
daily_spent = {account: 0.0 for account in initial_balances}
decisions = []
review_queue = []
# YOUR CODE HERE


In [56]:
required_decision_keys = {
    "id", "account", "kind", "amount",
    "status", "reason", "balance_after",
}
daily_limits = {
    "A100": 12_000.0,
    "A200": 8_000.0,
    "A300": 5_000.0,
    "A400": 3_000.0,
}
blocked_accounts = {"A400"}
blocked_merchants = {"SCAM-SHOP"}
reason_to_status = {
    "ok": "APPROVED",
    "daily_limit": "REVIEW",
    "blocked_account": "REJECTED",
    "blocked_merchant": "REJECTED",
    "unsupported_kind": "REJECTED",
    "insufficient_funds": "REJECTED",
}

#Решение
for dicts in clean_events:
    if dicts['kind'] in ('deposit', 'refund'):  #разбивка операций на дебит, кредит и неподдерживаемые
        direction = 'credit'
    elif dicts['kind'] in ('purchase', 'withdrawal', 'transfer'):
        direction = 'debit'
    else:
        direction = 'unsupported'
    status = None
    reason = None

    if direction == 'unsupported': #проверки
        status, reason = 'REJECTED', 'unsupported_kind'
    elif dicts['account'] in blocked_accounts:
        status, reason = 'REJECTED', 'blocked_account'
    elif direction == 'debit' and dicts['merchant'] in blocked_merchants:
        status, reason = 'REJECTED', 'blocked_merchant'
    elif direction == 'debit' and final_balances[dicts['account']] < dicts['amount']:
        status, reason = 'REJECTED', 'insufficient_funds'
    elif direction == 'debit' and daily_spent[dicts['account']] + dicts['amount'] > daily_limits[dicts['account']]:
        status, reason ='REVIEW', 'daily_limit'
    else:
        status, reason = 'APPROVED', 'ok'

    if status == 'APPROVED':
        if direction == 'credit':
            final_balances[dicts['account']] += dicts['amount']
        else:
            final_balances[dicts['account']] -= dicts['amount']
            daily_spent[dicts['account']] += dicts['amount']
            if dicts['kind'] == 'transfer':
                final_balances[dicts['counterparty']] += dicts['amount']

    decisions.append({'id': dicts['id'],
                    'account': dicts['account'],
                    'kind': dicts['kind'],
                    'amount': dicts['amount'],
                    'status': status,
                    'reason': reason,
                    'balance_after': final_balances[dicts['account']]})

    if status == 'REVIEW':
        review_queue.append(dicts.copy())

for i in decisions:
    print(i)

assert final_balances is not initial_balances
assert set(final_balances) == set(initial_balances)
assert set(daily_spent) == set(initial_balances)
assert all(amount >= 0 for amount in daily_spent.values())
assert len(decisions) == len(clean_events)
assert [item["id"] for item in decisions] == [event["id"] for event in clean_events]
assert all(set(item) == required_decision_keys for item in decisions)
assert all(
    reason_to_status.get(item["reason"]) == item["status"]
    for item in decisions
)

decision_review_ids = [
    item["id"] for item in decisions if item["status"] == "REVIEW"
]
assert [event["id"] for event in review_queue] == decision_review_ids
assert all(
    all(queued is not event for event in clean_events)
    for queued in review_queue
)
assert initial_balances == {
    "A100": 25_000.0,
    "A200": 15_000.0,
    "A300": 7_000.0,
    "A400": 5_000.0,
}
assert raw_events == raw_snapshot
print("Задание 4: журнал решений и инварианты состояния соблюдены")


{'id': 'E001', 'account': 'A100', 'kind': 'deposit', 'amount': 5000.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 30000.0}
{'id': 'E002', 'account': 'A100', 'kind': 'purchase', 'amount': 3200.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 26800.0}
{'id': 'E003', 'account': 'A200', 'kind': 'purchase', 'amount': 6500.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 8500.0}
{'id': 'E004', 'account': 'A200', 'kind': 'purchase', 'amount': 2500.0, 'status': 'REVIEW', 'reason': 'daily_limit', 'balance_after': 8500.0}
{'id': 'E005', 'account': 'A300', 'kind': 'withdrawal', 'amount': 1500.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 5500.0}
{'id': 'E006', 'account': 'A400', 'kind': 'purchase', 'amount': 1000.0, 'status': 'REJECTED', 'reason': 'blocked_account', 'balance_after': 5000.0}
{'id': 'E007', 'account': 'A100', 'kind': 'refund', 'amount': 700.0, 'status': 'APPROVED', 'reason': 'ok', 'balance_after': 27500.0}
{'id': 'E008', 'account': 'A100', 

## 5. Вложенная агрегация

Журнал решений удобен для аудита, но руководителю нужны итоги.
Словарь может содержать другие словари: например,
`stats_by_status[status]["amount"]`. Метод `setdefault` создаёт
начальное значение только при отсутствии ключа.


<!-- theory-formulas-v1 -->
### Теория: группировка, счётчик и сумма

Агрегация уменьшает набор подробных записей до показателей по группам. Для
статуса $s$ количество и сумма определяются как

$$
N_s=\sum_{i=1}^{n}\mathbf{1}[status_i=s],
$$

$$
A_s=\sum_{i=1}^{n}amount_i\,\mathbf{1}[status_i=s],
$$

где индикатор $\mathbf{1}[P]$ равен 1, когда условие $P$ истинно, и 0 иначе.

В коде индикатор реализуется условием, а аккумуляторы хранятся во вложенном
словаре:

```python
stats[status]["count"] += 1
stats[status]["amount"] += amount
```

Начальное состояние счётчика — 0, суммы — `0.0`, множества счетов — `set()`.
`dict.get(key, 0)` удобен для одного счётчика; `setdefault` — когда по ключу
нужно создать более сложный вложенный контейнер.


### Задание 5. Статистика решений — 2 балла

За один проход по `decisions` постройте:

- `stats_by_status`: для каждого статуса количество `count` и сумма
  `amount`;
- `reason_counts`: количество решений по каждой причине;
- `review_accounts`: множество счетов со статусом `REVIEW`;
- `rejected_accounts`: множество счетов со статусом `REJECTED`.

Не задавайте имена статусов заранее: структура должна заполняться
по данным.


**Уточнение контракта.** В `stats_by_status[status]["amount"]` суммируются
номинальные суммы **всех попыток** соответствующего статуса. Это не денежный
оборот применённых операций: суммы REVIEW и REJECTED также входят в свои группы.
`review_accounts` и `rejected_accounts` содержат уникальные счета, поэтому для
них используются множества.


In [57]:
stats_by_status = {}
reason_counts = {}
review_accounts = set()
rejected_accounts = set()
# YOUR CODE HERE


In [58]:
statuses_in_decisions = {item["status"] for item in decisions}
reasons_in_decisions = {item["reason"] for item in decisions}

#Решение
for dec in decisions:
    gr1 = stats_by_status.setdefault(dec['status'], {'count': 0, 'amount': 0.0}) #задаю ключи для словаря с нулями по количеству и сумме
    gr1['count'] += 1
    gr1['amount'] += dec['amount']
    reason_counts[dec['reason']] = reason_counts.get(dec['reason'], 0) + 1 #получаю ключ с начальным значением 0 и добавляю по единице для подсчета количества
    if dec['status'] == 'REVIEW':
        review_accounts.add(dec['account']) #добавляю во множество счета со статусом REVIEW
    elif dec['status'] == 'REJECTED':
        rejected_accounts.add(dec['account']) #аналогично добавляю счета со статусом REJECTED
print(stats_by_status)
print(reason_counts)
print(review_accounts)
print(rejected_accounts)

assert set(stats_by_status) == statuses_in_decisions
assert all(set(group) == {"count", "amount"} for group in stats_by_status.values())
assert sum(group["count"] for group in stats_by_status.values()) == len(decisions)
assert sum(group["amount"] for group in stats_by_status.values()) == sum(
    item["amount"] for item in decisions
)
assert set(reason_counts) == reasons_in_decisions
assert sum(reason_counts.values()) == len(decisions)
assert review_accounts == {
    item["account"] for item in decisions if item["status"] == "REVIEW"
}
assert rejected_accounts == {
    item["account"] for item in decisions if item["status"] == "REJECTED"
}
print("Задание 5: агрегаты согласованы с журналом решений")


{'APPROVED': {'count': 5, 'amount': 16900.0}, 'REVIEW': {'count': 2, 'amount': 6500.0}, 'REJECTED': {'count': 3, 'amount': 1950.0}}
{'ok': 5, 'daily_limit': 2, 'blocked_account': 1, 'blocked_merchant': 1, 'unsupported_kind': 1}
{'A300', 'A200'}
{'A400', 'A300', 'A100'}
Задание 5: агрегаты согласованы с журналом решений


## 6. Очередь и цикл `while`

`for` удобен, когда известна коллекция для обхода. `while` подходит,
когда завершение определяется состоянием: очередь пуста, исчерпан
лимит или достигнута контрольная точка.

Для учебного списка используем `pop(0)`. У больших очередей такая
операция дорогая; с `collections.deque` познакомимся в работе со
стандартной библиотекой.


<!-- theory-formulas-v1 -->
### Теория: FIFO, условие завершения и вариант цикла

Очередь FIFO обслуживает элементы в порядке поступления. Если

$$
Q_t=(e_0,e_1,\ldots,e_k),
$$

то извлечение головы даёт $e_0$, а новое состояние очереди

$$
Q_{t+1}=(e_1,\ldots,e_k).
$$

У цикла должны быть:

- **условие продолжения** — очередь не пуста;
- **инвариант** — уже обработанные элементы не появляются снова;
- **вариант** — величина, приближающаяся к завершению, здесь $|Q_t|$;
- **аварийная граница** — `break`, если исчерпан ручной лимит.

Рабочая очередь должна быть копией. При `pending = review_queue` оба имени
указывают на один список, и `pop(0)` разрушит исходный журнал. После `break`
текущий и ещё не просмотренные элементы должны остаться учтёнными как deferred,
иначе система молча потеряет заявки.


### Задание 6. Ручная проверка — 2 балла

Оператор может вручную одобрить превышения суммарно не более чем на
3000 RUB. Создайте копии `final_balances`, `daily_spent` и
`review_queue`.

Пока очередь не пуста:

- извлеките первый элемент;
- если его сумма превышает остаток ручного лимита, поместите ID
  текущего и всех оставшихся элементов в `deferred_review_ids` и
  завершите цикл через `break`;
- иначе примените списание или перевод, обновите дневной расход и
  лимит, добавьте ID в `manual_approved_ids`.

Исходная `review_queue` измениться не должна.


**Уточнение контракта.** Очередь обрабатывается строго FIFO: пропускать слишком
крупное первое событие и переходить к следующему нельзя. `manual_budget` —
остаток суммарного объёма операций, который оператор ещё может одобрить; после
одобрения он уменьшается на `event["amount"]`.

Ручное одобрение в этой задаче отменяет только причину `daily_limit`. Для
выданного набора у всех событий очереди достаточно денег, поэтому отдельного
сценария `insufficient_funds` здесь нет. Перевод при ручном одобрении по-прежнему
изменяет два баланса.

<details>
<summary>Подсказка 1 — условие цикла</summary>

Работайте с `pending_review`, пока список не пуст. На каждой итерации извлекайте
нулевой элемент. Проверяйте его сумму относительно текущего `manual_budget`.

</details>

<details>
<summary>Подсказка 2 — корректный выход через break</summary>

Перед `break` добавьте в `deferred_review_ids` ID текущего элемента, а затем ID
всех элементов, оставшихся в `pending_review`. В ветви одобрения обновите баланс,
расход и budget; для `transfer` не забудьте получателя.

</details>


In [59]:
settled_balances = final_balances.copy()
settled_spent = daily_spent.copy()
pending_review = [event.copy() for event in review_queue]
manual_budget = 3_000.0
manual_approved_ids = []
deferred_review_ids = []

#Решение
while pending_review:
    event = pending_review.pop(0)
    if event['amount'] > manual_budget:
        deferred_review_ids.append(event['id'])
        for remaining in pending_review:
            deferred_review_ids.append(remaining['id'])
        break
    settled_balances[event['account']] -= event['amount']
    settled_spent[event['account']] += event['amount']
    if event['kind'] == 'transfer':
        settled_balances[event['counterparty']] += event['amount']
    manual_budget -= event['amount']
    manual_approved_ids.append(event['id'])
for i in pending_review:
    print(i)
print()
print(settled_balances)
print()
print(settled_spent)


{'A100': 27500.0, 'A200': 6000.0, 'A300': 5500.0, 'A400': 5000.0}

{'A100': 3200.0, 'A200': 9000.0, 'A300': 1500.0, 'A400': 0.0}


In [60]:
review_ids_before_manual = [event["id"] for event in review_queue]

assert settled_balances is not final_balances
assert settled_spent is not daily_spent
assert set(settled_balances) == set(final_balances)
assert set(settled_spent) == set(daily_spent)
assert manual_budget >= 0
assert not (set(manual_approved_ids) & set(deferred_review_ids))

assert manual_approved_ids + deferred_review_ids == review_ids_before_manual

approved_manual_amount = sum(
    event["amount"]
    for event in review_queue
    if event["id"] in manual_approved_ids
)
assert abs((3_000.0 - manual_budget) - approved_manual_amount) < 1e-12
assert all(
    settled_spent[account] >= daily_spent[account]
    for account in daily_spent
)
assert [event["id"] for event in review_queue] == [
    item["id"] for item in decisions if item["status"] == "REVIEW"
]
print("Задание 6: FIFO-порядок и ручной лимит соблюдены")


Задание 6: FIFO-порядок и ручной лимит соблюдены


## 7. Сортировка и отчёт

`sorted` умеет сравнивать кортежи: сначала по первому элементу,
затем по второму. Это позволяет построить рейтинг без отдельной
функции-ключа. `enumerate(..., start=1)` добавляет место в рейтинге.


<!-- theory-formulas-v1 -->
### Теория: порядок и ранжирование

Рейтинг — перестановка исходных записей, для которой значения ключа идут по
убыванию:

$$
k(x_{(1)})\ge k(x_{(2)})\ge\dots\ge k(x_{(n)}).
$$

Python сравнивает кортежи лексикографически: сначала первый элемент, при
равенстве — второй. Поэтому пары `(spent, account)` можно отсортировать без
отдельной функции-ключа. Важно заранее определить правило равенства; иначе два
решения с одинаковым расходом могут располагаться неожиданно.

Типичная сложность сортировки — $O(n\log n)$, тогда как один проход для суммы
или счётчика — $O(n)$. Для маленького учебного пакета разница незаметна, но она
важна при миллионах событий.

`enumerate(ranking, start=1)` не меняет данные, а добавляет позицию. Форматирование
`{amount:,.2f}` отделяет представление суммы от её числового значения.


### Задание 7. Отчёт смены — 1 балл

Сформируйте `spending_ranking` — список кортежей `(account, spent)`
по убыванию расхода. Затем создайте строки с номером места и
объедините их переводами строк в `shift_report`.

Первая строка должна быть `Рейтинг дневного расхода:`. Суммы
форматируйте с двумя знаками после запятой и разделителем тысяч.


**Уточнение контракта.** Рейтинг строится по `settled_spent`, то есть после
ручной проверки. Каждый счёт должен встретиться ровно один раз. В выданных
данных равных расходов нет; правило разрешения равенства в этой работе не
оценивается. `ranking_lines` не содержит заголовок, а `shift_report` состоит из
заголовка и этих строк, соединённых символом `\n`.


In [61]:
spending_ranking = [(keys, values) for keys, values in settled_spent.items()]
ranking_lines = []
shift_report = ""

#Решение
separator = '\n'
spending_ranking = sorted(spending_ranking, key = lambda x: (-x[1], x[0]))

ranking_lines = [f'{i}. {account} {amount:,.2f}' for i, (account, amount) in enumerate(spending_ranking, start = 1)]

shift_report = "Рейтинг дневного расхода:" + separator + separator.join(ranking_lines)
print(ranking_lines)
print()
print(shift_report)

['1. A200 9,000.00', '2. A100 3,200.00', '3. A300 1,500.00', '4. A400 0.00']

Рейтинг дневного расхода:
1. A200 9,000.00
2. A100 3,200.00
3. A300 1,500.00
4. A400 0.00


In [62]:
assert type(spending_ranking) is list
assert all(type(item) is tuple and len(item) == 2 for item in spending_ranking)
assert {account for account, _ in spending_ranking} == set(settled_spent)
assert len(spending_ranking) == len(settled_spent)

ranked_amounts = [amount for _, amount in spending_ranking]
assert ranked_amounts == sorted(ranked_amounts, reverse=True)
assert len(ranking_lines) == len(spending_ranking)
assert all(
    line.startswith(f"{place}. ")
    for place, line in enumerate(ranking_lines, start=1)
)
assert shift_report.splitlines() == ["Рейтинг дневного расхода:"] + ranking_lines
print(shift_report)


Рейтинг дневного расхода:
1. A200 9,000.00
2. A100 3,200.00
3. A300 1,500.00
4. A400 0.00


## 8. Итоговый мини-проект — 5 баллов

Вторая смена — независимый набор данных. Здесь появляется
**velocity-лимит**: число одобренных списаний со счёта не должно
превышать заданный максимум. Его проверяют после дневного лимита,
но до достаточности баланса.

### Задание 8. Ночная смена

1. Убедитесь, что ID уникальны (`capstone_ids_unique`).
2. Создайте копию балансов, нулевые расходы и счётчики списаний.
3. Обработайте события по правилам задания 4, добавив velocity-
   проверку. Заблокированный merchant проверяется первым.
4. Перевод меняет два баланса, но расход и число списаний относятся
   только к отправителю.
5. Постройте `capstone_status_counts`, списки review/rejected ID,
   множество риск-счетов и `capstone_health` (`LOW`, если баланс
   меньше 3000 RUB, иначе `STABLE`).
6. Сформируйте итоговую строку `capstone_report` заданного формата.

В `capstone_decisions` достаточно ключей `id`, `account`, `status`,
`reason`, `amount`. Операции `REVIEW`/`REJECTED` баланс не меняют.


**Полный порядок правил.** Сначала проверяется запрещённый merchant. Затем через
`match` определяется направление операции. Кредит применяется сразу. Для
списания последовательно проверяются дневной лимит, velocity-лимит и достаточный
баланс; только после этого изменяется состояние.

| Условие | `status` | `reason` |
|---|---|---|
| операция применена | `APPROVED` | `ok` |
| превышен дневной лимит | `REVIEW` | `daily_limit` |
| превышен velocity-лимит | `REVIEW` | `velocity_limit` |
| merchant запрещён | `REJECTED` | `blocked_merchant` |
| вид операции не поддерживается | `REJECTED` | `unsupported_kind` |
| недостаточно средств | `REJECTED` | `insufficient_funds` |

`capstone_status_counts` имеет формат `{status: count}`.
`capstone_review_ids` и `capstone_rejected_ids` сохраняют порядок событий.
Риск-счёт — счёт, у которого есть хотя бы одно решение, отличное от
`APPROVED`. `capstone_health` имеет формат `{account: "LOW" | "STABLE"}`.

Шаблон итоговой строки:

```text
approved=<count> | review=<count> | rejected=<count> | total_balance=<amount> RUB | risk_accounts=<accounts>
```

`total_balance` форматируется с разделителем тысяч и двумя знаками после
запятой. Риск-счета сортируются по алфавиту и соединяются запятыми без пробелов.

<details>
<summary>Подсказка 1 — таблица трассировки</summary>

Перед кодированием пройдите события вручную. Для каждого ID запишите баланс,
дневной расход и debit count до операции, затем первое сработавшее правило и
состояние после него.

</details>

<details>
<summary>Подсказка 2 — итоговые структуры</summary>

Сначала полностью заполните `capstone_decisions`. После этого отдельным проходом
или comprehensions получите счётчики, списки ID, риск-счета и health. Строку
отчёта собирайте последней только из уже рассчитанных структур.

</details>


<!-- theory-formulas-v1 -->
### Теория: velocity-лимит и таблица трассировки

Дневной лимит ограничивает сумму, velocity-лимит — количество одобренных
списаний. Если $V_t(a)$ — число списаний счёта $a$ до события, то для
одобренного дебета

$$
V_{t+1}(a)=V_t(a)+1.
$$

Для кредита, REVIEW и REJECTED счётчик не меняется. Перед одобрением проверяется
проектируемое значение:

$$
V_t(a)+1\le V_{max}(a).
$$

Одновременно должны выполняться ограничения суммы и количества:

$$
S_t(a)+m\le L(a)
\quad\land\quad
V_t(a)+1\le V_{max}(a).
$$

Если спецификация проверяет дневной лимит раньше velocity, событие, нарушающее
оба правила, получает причину `daily_limit`. Для ручной проверки удобно вести
таблицу трассировки:

| шаг | ID | баланс до | расход до | count до | решение | баланс после |
|---:|---|---:|---:|---:|---|---:|

Заполняйте строку до запуска кода. Такое моделирование обнаруживает неверный
порядок правил и помогает объяснить результат на защите.


In [78]:
capstone_initial_balances = {"C100": 9_000.0, "C200": 4_000.0}
capstone_daily_limits = {"C100": 5_000.0, "C200": 2_500.0}
capstone_velocity_limits = {"C100": 2, "C200": 2}
capstone_blocked_merchants = {"CASINO"}
capstone_events = [
    {"id": "C001", "kind": "purchase", "account": "C100",
     "amount": 1200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C002", "kind": "withdrawal", "account": "C200",
     "amount": 500.0, "merchant": None, "counterparty": None},
    {"id": "C003", "kind": "purchase", "account": "C100",
     "amount": 4100.0, "merchant": "TECH", "counterparty": None},
    {"id": "C004", "kind": "refund", "account": "C200",
     "amount": 200.0, "merchant": "BOOKS", "counterparty": None},
    {"id": "C005", "kind": "purchase", "account": "C200",
     "amount": 300.0, "merchant": "CASINO", "counterparty": None},
    {"id": "C006", "kind": "transfer", "account": "C200",
     "amount": 1800.0, "merchant": None, "counterparty": "C100"},
    {"id": "C007", "kind": "purchase", "account": "C200",
     "amount": 100.0, "merchant": "CAFE", "counterparty": None},
    {"id": "C008", "kind": "deposit", "account": "C100",
     "amount": 1000.0, "merchant": None, "counterparty": None},
]


In [85]:
capstone_ids_unique = None
capstone_balances = capstone_initial_balances.copy()
capstone_spent = {account: 0.0 for account in capstone_balances}
capstone_debit_counts = {account: 0 for account in capstone_balances}
capstone_decisions = []

capstone_status_counts = {}
capstone_review_ids = []
capstone_rejected_ids = []
capstone_risk_accounts = set()
capstone_health = {}
capstone_report = ""

#Решение
capstone_ids_unique = {dicts['id'] for dicts in capstone_events} #Множество уникальных id

for dicts in capstone_events:
    if dicts['merchant'] in capstone_blocked_merchants: #Проверки
        status, reason = 'REJECTED', 'blocked_merchant'
    else:
        if dicts['kind'] in ('deposit', 'refund'):
            direction = 'credit'
        elif dicts['kind'] in ('purchase', 'withdrawal', 'transfer'):
            direction = 'debit'
        else:
            direction = 'unsupported'

        if direction == 'unsupported':
            status, reason = 'REJECTED', 'unsupported_kind'
        elif direction == 'credit':
            status, reason = 'APPROVED', 'ok'
        else:
            if capstone_spent[dicts['account']] + dicts['amount'] > capstone_daily_limits[dicts['account']]:
                status, reason = 'REVIEW', 'daily_limit'
            elif capstone_debit_counts[dicts['account']] + 1 > capstone_velocity_limits[dicts['account']]:
                status, reason = 'REVIEW', 'velocity_limit'
            elif capstone_balances[dicts['account']] < dicts['amount']:
                status, reason = 'REJECTED', 'insufficient_funds'
            else:
                status, reason = 'APPROVED', 'ok'
    if status == 'APPROVED':
        if direction == 'credit':
            capstone_balances[dicts['account']] += dicts['amount']
        else:
            capstone_balances[dicts['account']] -= dicts['amount']
            capstone_spent[dicts['account']] += dicts['amount']
            capstone_debit_counts[dicts['account']] += 1
            if dicts['kind'] == 'transfer':
                capstone_balances[dicts['counterparty']] += dicts['amount']

    capstone_decisions.append({ #Добавление решений
        'id': dicts['id'],
        'account': dicts['account'],
        'status': status,
        'reason': reason,
        'amount': dicts['amount']
    })

for item in capstone_decisions: #Подсчет количества статусов
    capstone_status_counts[item['status']] = capstone_status_counts.get(item['status'], 0) + 1

print(capstone_status_counts)
capstone_review_ids = [item['id'] for item in capstone_decisions if item['status'] == 'REVIEW'] #id на рассмотрении
capstone_rejected_ids = [item['id'] for item in capstone_decisions if item['status'] == 'REJECTED'] #id отклоненные

capstone_risk_accounts = {item['account'] for item in capstone_decisions if item['status'] != 'APPROVED'}
capstone_health = {account: ('LOW' if balance < 3000 else 'STABLE') for account, balance in capstone_balances.items()}

total_balance = sum(capstone_balances.values())
approved_account = capstone_status_counts.get('APPROVED', 0)
review_count = capstone_status_counts.get('REVIEW', 0)
rejected_count = capstone_status_counts.get('REJECTED', 0)
risk_str = ','.join(sorted(capstone_risk_accounts))

capstone_report = (
    f'approved={approved_account} | '
    f'review={review_count} | '
    f'rejected={rejected_count} | '
    f'total_balance={total_balance:,.2f} RUB | '
    f'risk_accounts={risk_str}'
)

print(len(capstone_decisions), len(capstone_events))
print(capstone_report)

{'APPROVED': 5, 'REVIEW': 2, 'REJECTED': 1}
8 8
approved=5 | review=2 | rejected=1 | total_balance=12,500.00 RUB | risk_accounts=C100,C200


In [87]:
capstone_ids = [event["id"] for event in capstone_events]
required_capstone_keys = {"id", "account", "status", "reason", "amount"}
capstone_reason_to_status = {
    "ok": "APPROVED",
    "daily_limit": "REVIEW",
    "velocity_limit": "REVIEW",
    "blocked_merchant": "REJECTED",
    "unsupported_kind": "REJECTED",
    "insufficient_funds": "REJECTED",
}

#assert capstone_ids_unique == (len(capstone_ids) == len(set(capstone_ids)))
assert capstone_balances is not capstone_initial_balances
assert set(capstone_balances) == set(capstone_initial_balances)
assert set(capstone_spent) == set(capstone_initial_balances)
assert set(capstone_debit_counts) == set(capstone_initial_balances)
assert len(capstone_decisions) == len(capstone_events)
assert [item["id"] for item in capstone_decisions] == capstone_ids
assert all(set(item) == required_capstone_keys for item in capstone_decisions)
assert all(
    capstone_reason_to_status.get(item["reason"]) == item["status"]
    for item in capstone_decisions
)

expected_status_counts = {
    status: sum(item["status"] == status for item in capstone_decisions)
    for status in {item["status"] for item in capstone_decisions}
}

assert capstone_status_counts == expected_status_counts
assert capstone_review_ids == [
    item["id"] for item in capstone_decisions if item["status"] == "REVIEW"
]
assert capstone_rejected_ids == [
    item["id"] for item in capstone_decisions if item["status"] == "REJECTED"
]
assert capstone_risk_accounts == {
    item["account"]
    for item in capstone_decisions
    if item["status"] != "APPROVED"
}
assert capstone_health == {
    account: ("LOW" if balance < 3_000 else "STABLE")
    for account, balance in capstone_balances.items()
}

report_parts = capstone_report.split(" | ")
assert report_parts == [
    f"approved={capstone_status_counts.get('APPROVED', 0)}",
    f"review={capstone_status_counts.get('REVIEW', 0)}",
    f"rejected={capstone_status_counts.get('REJECTED', 0)}",
    f"total_balance={sum(capstone_balances.values()):,.2f} RUB",
    f"risk_accounts={','.join(sorted(capstone_risk_accounts))}",
]
print(capstone_report)
print(report_parts)

approved=5 | review=2 | rejected=1 | total_balance=12,500.00 RUB | risk_accounts=C100,C200
['approved=5', 'review=2', 'rejected=1', 'total_balance=12,500.00 RUB', 'risk_accounts=C100,C200']


## Вывод и защита

Добавьте после этой ячейки собственный вывод на 7–10 предложений:

1. Почему порядок событий влияет на результат?
2. Почему E004 отправлена на REVIEW, хотя денег на счёте достаточно?
3. Почему E009 не изменила два баланса автоматически?
4. Чем ошибка входных данных отличается от `REJECTED`?
5. Почему для `review_queue` понадобилась копия?
6. Как сработал velocity-лимит во второй смене?
7. Назовите хотя бы один инвариант системы.

### Контрольные вопросы

1. Когда нужен список, а когда множество?
2. Почему ключ словаря должен быть хешируемым?
3. Чем `append` отличается от `extend`?
4. Почему опасно удалять элементы списка внутри `for` по нему?
5. Когда `continue` делает код понятнее?
6. Чем `while` отличается от `for` в задаче с очередью?
7. Как `match` и `if` разделяют разные виды решений?
8. Что означает «не изменять исходные данные»?
9. Почему проверка лимита должна происходить до изменения баланса?
10. Когда comprehension хуже обычного цикла?


_Напишите здесь собственный вывод._
